# 모듈 2: 다층 퍼셉트론(MLP) 실습 및 증명

이 노트북에서는 파이토치를 활용하여
1. **코드 증명 A:** 활성화 함수 없이 층을 쌓으면 왜 무의미한지 (선형 붕괴) 증명
2. **코드 증명 B:** 시그모이드의 기울기 소실 문제를 실제 수치로 확인
3. **코드 증명 C:** Sigmoid vs ReLU 기울기 크기 비교 실험
4. 파이토치 내장 `nn.Linear`가 행렬곱과 본질적으로 같음을 증명
5. **은닉층의 순전파(Forward Pass) 계산 흐름을 단계별로 추적**
6. 은닉층(Hidden Layer) 도입으로 XOR 문제가 해결되는 과정을 확인합니다.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

---
## 코드 증명 A: 활성화 함수 없이 층을 쌓으면 의미가 없다 (선형 붕괴)

활성화 함수 없이 `nn.Linear`만 100층을 쌓아도 수학적으로 1층짜리 $y = Wx + b$와 완전히 같다는 것을 증명합니다.

**수학적 배경:** $y = W_2(W_1 x + b_1) + b_2 = (W_2 W_1)x + (W_2 b_1 + b_2) = W'x + b'$

In [ ]:
# ==============================================================
# 실험: 활성화 함수 없이 nn.Linear를 100층 쌓기
# ==============================================================
torch.manual_seed(42)

# 100층짜리 네트워크 (활성화 함수 없음!)
layers_no_act = []
for _ in range(100):
    layers_no_act.append(nn.Linear(4, 4))

deep_no_activation = nn.Sequential(*layers_no_act)

# 1층짜리 네트워크
single_layer = nn.Linear(4, 4)

# 테스트 입력
x_test = torch.randn(1, 4)

# 100층 네트워크의 전체 가중치를 하나로 합성
# W_total = W_100 × W_99 × ... × W_1
W_합성 = torch.eye(4)
b_합성 = torch.zeros(4)
for layer in layers_no_act:
    W_합성 = layer.weight @ W_합성
    b_합성 = layer.weight @ b_합성 + layer.bias

# 합성된 단일 가중치로 계산
output_합성 = x_test @ W_합성.T + b_합성

# 100층 네트워크로 직접 계산
with torch.no_grad():
    output_100층 = deep_no_activation(x_test)

print("=" * 60)
print("코드 증명 A: 활성화 함수 없는 100층 = 단일 선형 변환")
print("=" * 60)
print(f"100층 네트워크 출력:    {output_100층.detach().tolist()}")
print(f"합성 단일 변환 출력:    {output_합성.detach().tolist()}")
print(f"\n완벽 일치: {torch.allclose(output_100층, output_합성, atol=1e-3)}")
print("\n→ 결론: 활성화 함수 없이 아무리 많은 층을 쌓아도")
print("  결국 y = W'x + b' 하나의 1차 변환으로 축약됩니다.")
print("  층을 깊게 쌓는 의미가 완전히 사라집니다!")

---
## 코드 증명 B: 시그모이드의 기울기 소실 (Gradient Vanishing) 실험

시그모이드를 은닉층에 사용하면 깊은 층의 기울기가 얼마나 작아지는지 확인합니다.

시그모이드 미분의 최대값은 0.25 → 층마다 곱해지면 $0.25^n$으로 급속 감소!

In [ ]:
# ==============================================================
# 실험: 5층 네트워크에서 시그모이드의 기울기 소실 관찰
# ==============================================================
torch.manual_seed(42)

class SigmoidNet(nn.Module):
    """시그모이드를 은닉층에 사용하는 5층 네트워크"""
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 8)
        self.layer2 = nn.Linear(8, 8)
        self.layer3 = nn.Linear(8, 8)
        self.layer4 = nn.Linear(8, 8)
        self.layer5 = nn.Linear(8, 1)
    
    def forward(self, x):
        x = torch.sigmoid(self.layer1(x))  # 은닉층 1
        x = torch.sigmoid(self.layer2(x))  # 은닉층 2
        x = torch.sigmoid(self.layer3(x))  # 은닉층 3
        x = torch.sigmoid(self.layer4(x))  # 은닉층 4
        x = self.layer5(x)                 # 출력층
        return x

sigmoid_net = SigmoidNet()

# 순전파 + 역전파 실행
x_input = torch.randn(1, 4)
y_target = torch.tensor([[1.0]])
output = sigmoid_net(x_input)
loss = (output - y_target) ** 2
loss.backward()

print("=" * 60)
print("코드 증명 B: 시그모이드 → 기울기 소실 관찰")
print("=" * 60)
print(f"{'레이어':>10} | {'기울기 평균 크기':>18} | {'상태':>10}")
print("-" * 45)

for name, param in sigmoid_net.named_parameters():
    if 'weight' in name:
        grad_magnitude = param.grad.abs().mean().item()
        status = "⚠️ 소실!" if grad_magnitude < 0.001 else "보통" if grad_magnitude < 0.01 else "양호"
        print(f"{name:>10} | {grad_magnitude:>18.8f} | {status:>10}")

print(f"\n→ 관찰: 출력에서 멀어질수록(layer1 방향) 기울기가 급격히 작아집니다.")
print(f"  → 앞쪽 레이어는 학습이 거의 이루어지지 않습니다!")

---
## 코드 증명 C: Sigmoid vs ReLU 기울기 크기 비교

동일한 5층 구조에서 활성화 함수만 ReLU로 교체했을 때 기울기가 어떻게 달라지는지 확인합니다.

In [ ]:
# ==============================================================
# 실험: ReLU를 사용하는 동일한 5층 구조
# ==============================================================
torch.manual_seed(42)

class ReLUNet(nn.Module):
    """ReLU를 은닉층에 사용하는 5층 네트워크"""
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 8)
        self.layer2 = nn.Linear(8, 8)
        self.layer3 = nn.Linear(8, 8)
        self.layer4 = nn.Linear(8, 8)
        self.layer5 = nn.Linear(8, 1)
    
    def forward(self, x):
        x = torch.relu(self.layer1(x))   # ReLU로만 교체!
        x = torch.relu(self.layer2(x))
        x = torch.relu(self.layer3(x))
        x = torch.relu(self.layer4(x))
        x = self.layer5(x)
        return x

relu_net = ReLUNet()

# 동일한 입력으로 순전파 + 역전파
output_relu = relu_net(x_input)
loss_relu = (output_relu - y_target) ** 2
loss_relu.backward()

# ============== 비교 출력 ==============
print("=" * 70)
print("코드 증명 C: Sigmoid vs ReLU 기울기 크기 비교")
print("=" * 70)
print(f"{'레이어':>10} | {'Sigmoid 기울기':>18} | {'ReLU 기울기':>18} | {'배율':>8}")
print("-" * 62)

sigmoid_grads = {}
relu_grads = {}

for name, param in sigmoid_net.named_parameters():
    if 'weight' in name:
        sigmoid_grads[name] = param.grad.abs().mean().item()

for name, param in relu_net.named_parameters():
    if 'weight' in name:
        relu_grads[name] = param.grad.abs().mean().item()

for name in sigmoid_grads:
    sig_g = sigmoid_grads[name]
    rel_g = relu_grads[name]
    ratio = rel_g / sig_g if sig_g > 0 else float('inf')
    print(f"{name:>10} | {sig_g:>18.8f} | {rel_g:>18.8f} | {ratio:>7.1f}×")

print(f"\n→ 결론: ReLU는 특히 앞쪽 레이어(layer1)에서 Sigmoid 대비")
print(f"  수십~수백 배 큰 기울기를 유지합니다.")
print(f"  → 깊은 네트워크에서도 모든 층이 골고루 학습됩니다!")

In [ ]:
# ==============================================================
# 시각화: 레이어별 기울기 크기 막대 그래프
# ==============================================================
layer_names = [n.replace('.weight', '') for n in sigmoid_grads.keys()]
sig_values = list(sigmoid_grads.values())
rel_values = list(relu_grads.values())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 절대값 비교
x_pos = range(len(layer_names))
axes[0].bar([x - 0.2 for x in x_pos], sig_values, 0.35, label='Sigmoid', color='blue', alpha=0.7)
axes[0].bar([x + 0.2 for x in x_pos], rel_values, 0.35, label='ReLU', color='red', alpha=0.7)
axes[0].set_xlabel('Layer', fontsize=12)
axes[0].set_ylabel('Gradient Magnitude', fontsize=12)
axes[0].set_title('레이어별 기울기 크기 비교', fontsize=14)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(layer_names)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# 기울기 비율(ReLU / Sigmoid) 비교
ratios = [r / s if s > 0 else 0 for r, s in zip(rel_values, sig_values)]
colors = ['green' if r > 1 else 'gray' for r in ratios]
axes[1].bar(layer_names, ratios, color=colors, alpha=0.7)
axes[1].axhline(y=1, color='black', linestyle='--', alpha=0.5, label='동일 기준선')
axes[1].set_xlabel('Layer', fontsize=12)
axes[1].set_ylabel('ReLU / Sigmoid 배율', fontsize=12)
axes[1].set_title('ReLU 기울기가 Sigmoid의 몇 배인가?', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("→ 왼쪽: Sigmoid(파랑)는 앞쪽 레이어 기울기가 거의 보이지 않을 만큼 작다")
print("→ 오른쪽: ReLU는 앞쪽 레이어에서 Sigmoid의 수십~수백배 기울기를 유지한다")

---
## 1. 활성화 함수 모양 비교 (Step vs Sigmoid vs ReLU)
계단 함수의 미분 불가능성 문제를 부드럽게 해결하기 위해 도입된 시그모이드와, 
그 시그모이드의 치명적인 단점인 '기울기 소실' 문제를 극복한 직관적인 ReLU의 형태를 시각적으로 확인합니다.

In [ ]:
x_vals = torch.linspace(-5, 5, 100)

sigmoid_fn = nn.Sigmoid()
relu_fn = nn.ReLU()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# --- 상단: 함수 형태 ---
# Step Function
step_y = (x_vals >= 0).float()
axes[0][0].plot(x_vals.numpy(), step_y.numpy(), color='gray', linewidth=2)
axes[0][0].set_title('Step Function', fontsize=12)
axes[0][0].grid(True, alpha=0.3)

# Sigmoid
axes[0][1].plot(x_vals.numpy(), sigmoid_fn(x_vals).numpy(), color='blue', linewidth=2)
axes[0][1].set_title('Sigmoid', fontsize=12)
axes[0][1].grid(True, alpha=0.3)

# ReLU
axes[0][2].plot(x_vals.numpy(), relu_fn(x_vals).numpy(), color='red', linewidth=2)
axes[0][2].set_title('ReLU', fontsize=12)
axes[0][2].grid(True, alpha=0.3)

# --- 하단: 미분(기울기) 형태 ---
# Step 미분: 거의 항상 0
step_grad = torch.zeros_like(x_vals)
axes[1][0].plot(x_vals.numpy(), step_grad.numpy(), color='gray', linewidth=2)
axes[1][0].set_title('Step 미분: 항상 0 (학습불가)', fontsize=11, color='red')
axes[1][0].set_ylim(-0.1, 0.5)
axes[1][0].grid(True, alpha=0.3)

# Sigmoid 미분: σ(x)(1-σ(x)), 최대 0.25
sig_y = sigmoid_fn(x_vals)
sig_grad = sig_y * (1 - sig_y)
axes[1][1].plot(x_vals.numpy(), sig_grad.numpy(), color='blue', linewidth=2)
axes[1][1].axhline(y=0.25, color='orange', linestyle='--', alpha=0.7, label='최대 0.25')
axes[1][1].set_title('Sigmoid 미분: 최대 0.25 (소실 위험)', fontsize=11, color='orange')
axes[1][1].legend()
axes[1][1].set_ylim(-0.1, 0.5)
axes[1][1].grid(True, alpha=0.3)

# ReLU 미분: 양수 구간에서 항상 1
relu_grad = (x_vals > 0).float()
axes[1][2].plot(x_vals.numpy(), relu_grad.numpy(), color='red', linewidth=2)
axes[1][2].set_title('ReLU 미분: 양수에서 1 (기울기 유지!)', fontsize=11, color='green')
axes[1][2].set_ylim(-0.1, 1.5)
axes[1][2].grid(True, alpha=0.3)

plt.suptitle('활성화 함수의 형태(상단)와 미분값(하단) 비교', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("→ 핵심: 하단의 미분(기울기) 그래프가 학습 능력을 결정합니다")
print("  Step: 기울기 0 → 학습 불가")
print("  Sigmoid: 기울기 최대 0.25 → 깊어지면 소실")
print("  ReLU: 기울기 1 → 깊어져도 유지!")

---
## 2. nn.Linear의 내부 행렬 연산 완벽 일치 증명

파이토치의 `nn.Linear`는 블랙박스가 아닙니다. 
내부 작동 방식이 우리가 배운 수식 $y = xW^T + b$와 정확히 같음을 코드로 검증합니다.

In [ ]:
입력 = torch.tensor([[1.0, 2.0]])

# 1. PyTorch 기본 모듈 사용
linear_layer = nn.Linear(in_features=2, out_features=3)
pytorch_결과 = linear_layer(입력)

# 2. 직접 수작업 행렬 연산 구현
W = linear_layer.weight.T  # nn.Linear는 weight를 전치(Transpose) 형태로 저장
b = linear_layer.bias
수작업_결과 = torch.matmul(입력, W) + b

print("PyTorch nn.Linear 연산 결과:", pytorch_결과.detach())
print("수작업 행렬 연산 결과:      ", 수작업_결과.detach())
print(f"\n완벽 일치 여부: {torch.allclose(pytorch_결과, 수작업_결과)}")
print("-> nn.Linear = 행렬곱 + 편향 덧셈, 그 이상도 이하도 아닙니다.")

---
## 3. ⭐ 은닉층 순전파(Forward Pass) 단계별 계산 흐름 추적

이 섹션이 모듈 2의 **핵심**입니다.
XOR 문제를 풀기 위한 2층 신경망의 순전파를 4단계로 분해하여,
각 단계에서 텐서(숫자)가 어떻게 변환되는지 하나하나 따라갑니다.

```
 입력(x)  →  ① 선형결합  →  ② 활성화  →  ③ 선형결합  →  ④ 활성화  →  출력(ŷ)
 [2개]       z⁽¹⁾=[2개]    h=[2개]      z⁽²⁾=[1개]    ŷ=[1개]
```

In [ ]:
# ========================================================
# XOR 데이터 정의
# ========================================================
X = torch.tensor([[0.0, 0.0],   # 정답: 0
                  [0.0, 1.0],   # 정답: 1
                  [1.0, 0.0],   # 정답: 1
                  [1.0, 1.0]])  # 정답: 0

# ========================================================
# 미리 계산된 XOR 해결 가중치 세팅
# (나중에 경사하강법을 배우면 이 값을 자동으로 찾게 됩니다)
# ========================================================
W1 = torch.tensor([[20.0, -20.0],
                   [20.0, -20.0]])
b1 = torch.tensor([-10.0, 30.0])

W2 = torch.tensor([[20.0],
                   [20.0]])
b2 = torch.tensor([-30.0])

print("가중치 설정 완료")
print(f"W1(입력→은닉) shape: {W1.shape}  |  b1: {b1.tolist()}")
print(f"W2(은닉→출력) shape: {W2.shape}  |  b2: {b2.tolist()}")

In [ ]:
# ========================================================
# 순전파 단계별 추적 (입력 [1.0, 0.0]을 예시로)
# ========================================================
x_sample = X[2]  # [1.0, 0.0] -> XOR 정답은 1
print(f"입력 데이터: {x_sample.tolist()}")
print(f"XOR 정답: 1")
print("=" * 55)

# ① 은닉층 선형결합: z = x·W + b
z1 = torch.matmul(x_sample, W1) + b1
print(f"\n① 은닉층 선형결합 z⁽¹⁾ = x·W₁ + b₁")
print(f"   = {x_sample.tolist()} · {W1.tolist()} + {b1.tolist()}")
print(f"   = {z1.tolist()}")

# ② 은닉층 활성화 (시그모이드)
h = torch.sigmoid(z1)
print(f"\n② 은닉층 활성화 h = sigmoid(z⁽¹⁾)")
print(f"   = sigmoid({z1.tolist()})")
print(f"   = {[round(v, 4) for v in h.tolist()]}")
print(f"   → 10이라는 큰 양수가 시그모이드를 통과하면 ≈1.0 (거의 확실히 활성화)")

# ③ 출력층 선형결합
z2 = torch.matmul(h, W2) + b2
print(f"\n③ 출력층 선형결합 z⁽²⁾ = h·W₂ + b₂")
print(f"   = {[round(v, 4) for v in h.tolist()]} · {W2.tolist()} + {b2.tolist()}")
print(f"   = {round(z2.item(), 4)}")

# ④ 출력층 활성화 (최종 확률)
y_hat = torch.sigmoid(z2)
print(f"\n④ 출력 활성화 ŷ = sigmoid(z⁽²⁾)")
print(f"   = sigmoid({round(z2.item(), 4)})")
print(f"   = {round(y_hat.item(), 4)}")
print(f"   → 판정: {1 if y_hat.item() > 0.5 else 0} (정답: 1) ✅")

In [ ]:
# ========================================================
# 모든 XOR 입력에 대해 전체 순전파 한번에 실행 (행렬 연산의 위력)
# ========================================================
print("=" * 60)
print("전체 XOR 입력에 대한 순전파 결과")
print("=" * 60)

# 행렬 연산으로 4개 입력을 한번에 처리!
Z1_all = torch.matmul(X, W1) + b1       # ① 은닉층 선형결합
H_all  = torch.sigmoid(Z1_all)           # ② 은닉층 활성화
Z2_all = torch.matmul(H_all, W2) + b2   # ③ 출력층 선형결합
Y_all  = torch.sigmoid(Z2_all)           # ④ 출력 활성화

xor_targets = [0, 1, 1, 0]
for i, (x_val, h_val, y_val) in enumerate(zip(X, H_all, Y_all)):
    target = xor_targets[i]
    pred = 1 if y_val.item() > 0.5 else 0
    status = "✅" if pred == target else "❌"
    print(f"입력 {x_val.tolist()} → 은닉층 출력 {[round(v, 4) for v in h_val.tolist()]} "
          f"→ 최종 확률 {y_val.item():.4f} → 판정 {pred} (정답 {target}) {status}")

print("\n→ 결과 분석: 은닉층 도입과 시그모이드를 통한 비선형 곡률 덕분에")
print("  불가능했던 XOR 분리에 완벽히 성공했습니다!")

---
## 4. 핵심 정리: 왜 이 흐름을 반드시 이해해야 하는가?

| 개념 | MLP (지금 배운 것) | RNN (나중에 배울 것) |
|:---:|:---:|:---:|
| 순전파 횟수 | **1회** (입력→은닉→출력) | **시간 단계(t)만큼 반복** |
| 은닉 상태 | 독립적 | 이전 시간의 h를 다음 시간에 재활용 |
| 핵심 연산 | `z = x·W + b` → `h = σ(z)` | `h_t = σ(x_t·W + h_{t-1}·U + b)` |

MLP의 순전파 한 사이클(①→②→③→④)을 체화하면,  
RNN은 "그것을 시간축으로 펼쳐놓은 것"에 불과합니다.

### 활성화 함수 선택 요약

| 위치 | 추천 함수 | 이유 |
|:---:|:---:|:---|
| **은닉층** | ReLU | 기울기 유지(=1), 빠른 연산 |
| **이진 분류 출력층** | Sigmoid | 0~1 확률 출력 |
| **다중 분류 출력층** | Softmax | 확률 합 = 1 |
| **회귀 출력층** | 없음 | 연속 실수값 그대로 |

## 6. 공식 적용 예제: XOR를 MLP로 해결

단층 퍼셉트론으로 불가능한 XOR을 MLP 공식으로 해결하는 예제입니다.

- 은닉층: $h=ReLU(W_1x+b_1)$
- 출력층: $\hat{y}=\sigma(W_2h+b_2)$

비선형 은닉층이 들어가면 선형 분리가 안 되는 문제도 분리 가능합니다.


In [ ]:
import torch
import torch.nn as nn

X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y = torch.tensor([[0.],[1.],[1.],[0.]])

model = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)
loss_fn = nn.BCELoss()
opt = torch.optim.Adam(model.parameters(), lr=0.05)

for _ in range(2000):
    pred = model(X)
    loss = loss_fn(pred, y)
    opt.zero_grad(); loss.backward(); opt.step()

with torch.no_grad():
    out = model(X)
    print('예측 확률:', out.squeeze().tolist())
    print('분류 결과:', (out >= 0.5).int().squeeze().tolist())
